In [1]:
%load_ext sql

# Connect SQLMagic directly to the project DuckDB file.
%sql duckdb:///f:/Project/games/jt3/data/jt3.duckdb

%config SqlMagic.autopolars = True

%config SqlMagic.named_parameters = "enabled"

The 'toml' package isn't installed. To load settings from pyproject.toml or ~/.jupysql/config, install with: pip install toml

Connecting to 'duckdb:///f:/Project/games/jt3/data/jt3.duckdb'

In [2]:
%%sql rows <<
SELECT DISTINCT c.text AS clue_text
FROM clues AS c
ORDER BY c.game_id DESC, c.round_index, c.category_index, c.clue_order

Running query in 'duckdb:///f:/Project/games/jt3/data/jt3.duckdb'

In [3]:
from sentence_transformers import SentenceTransformer

MODELS = {
    "all_MiniLM_L6_v2": dict(model_name_or_path="sentence-transformers/all-MiniLM-L6-v2", device='cuda', backend="onnx", model_kwargs={"provider": "CUDAExecutionProvider"}),
    "qwen3_embedding_06B_trunc_32": dict(model_name_or_path="Qwen/Qwen3-Embedding-0.6B", device='cuda', truncate_dim=32),
    "qwen3_embedding_06B": dict(model_name_or_path="Qwen/Qwen3-Embedding-0.6B", device='cuda'),
}

model = SentenceTransformer(**MODELS["qwen3_embedding_06B"])

In [ ]:
clues = rows['clue_text'].to_list()
embeddings = model.encode(clues, batch_size=128, show_progress_bar=True)

Batches:   0%|          | 0/576 [00:00<?, ?it/s]

In [ ]:
import polars as pl
import duckdb

dim = embeddings.shape[1]
df = pl.DataFrame({
    "clue_text": clues,
    "embedding": embeddings.tolist(),
}).unique(subset=["clue_text"], keep="first")

# Close SQLMagic's connection so we can get a write lock with native duckdb.
%sql --close duckdb:///f:/Project/games/jt3/data/jt3.duckdb

con = duckdb.connect("f:/Project/games/jt3/data/jt3.duckdb")
con.sql("DROP TABLE IF EXISTS clue_embeddings")
con.sql(f"CREATE TABLE clue_embeddings (clue_text TEXT PRIMARY KEY, embedding FLOAT[{dim}] NOT NULL)")
con.sql("INSERT INTO clue_embeddings SELECT clue_text, embedding FROM df")
con.close()

print(f"Saved {len(df)} clue embeddings ({len(clues) - len(df)} duplicates removed)")

In [ ]:
# Reopen for subsequent %sql cells.
%sql duckdb:///f:/Project/games/jt3/data/jt3.duckdb

%sql SELECT * FROM clue_embeddings LIMIT 5

In [ ]:
%%sql response_rows <<
SELECT DISTINCT c.correct_response AS response_text
FROM clues AS c
WHERE c.correct_response IS NOT NULL
ORDER BY c.game_id DESC, c.round_index, c.category_index, c.clue_order

In [ ]:
responses = response_rows['response_text'].to_list()
response_embeddings = model.encode(responses, batch_size=128, show_progress_bar=True)

In [ ]:
rdim = response_embeddings.shape[1]
rdf = pl.DataFrame({
    "response_text": responses,
    "embedding": response_embeddings.tolist(),
}).unique(subset=["response_text"], keep="first")

%sql --close duckdb:///f:/Project/games/jt3/data/jt3.duckdb

con = duckdb.connect("f:/Project/games/jt3/data/jt3.duckdb")
con.sql("DROP TABLE IF EXISTS response_embeddings")
con.sql(f"CREATE TABLE response_embeddings (response_text TEXT PRIMARY KEY, embedding FLOAT[{rdim}] NOT NULL)")
con.sql("INSERT INTO response_embeddings SELECT response_text, embedding FROM rdf")
con.close()

print(f"Saved {len(rdf)} response embeddings ({len(responses) - len(rdf)} duplicates removed)")

In [ ]:
%sql duckdb:///f:/Project/games/jt3/data/jt3.duckdb

%sql SELECT * FROM response_embeddings LIMIT 5